# Tutorial 4: The AI Data Factory (Surrogate Modeling)

Physics simulations are highly accurate but can be computationally expensive 
for massive datasets. In modern materials science, researchers train 
Machine Learning "surrogate models" to predict physical properties instantly.

To train these models, you need data. In this tutorial, we will use OpenImpala 
as an automated "Data Factory". We will:
1. Generate 50 randomized microstructures.
2. Measure their Morphological properties (Porosity, Surface Area).
3. Use OpenImpala to calculate their "Ground Truth" Tortuosity.
4. Train a Random Forest model to predict Tortuosity 1000x faster.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import porespy as ps
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import openimpala as oi

N_SAMPLES = 50
SIZE = 32

In [ ]:
print("--- Step 1: Generating Data with OpenImpala ---")
X_features = [] # Will hold [Porosity, Specific Surface Area]
y_target = []   # Will hold [Tortuosity]

# Wrap the entire generation loop in one Session to avoid MPI init/finalize overhead
with oi.Session():
    for i in range(N_SAMPLES):
        # 1. Generate random blobby structure (varying porosity and blob size)
        target_porosity = np.random.uniform(0.3, 0.7)
        blobiness = np.random.uniform(1.0, 3.0)
        micro = ps.generators.blobs(shape=[SIZE, SIZE, SIZE], porosity=target_porosity, blobiness=blobiness)
        micro = micro.astype(np.int32)
        
        # 2. Extract quick morphological features
        vf = oi.volume_fraction(micro, phase=1).fraction
        surface_area = ps.metrics.specific_surface_area(micro)
        
        # 3. Get Ground Truth physics from OpenImpala
        perc = oi.percolation_check(micro, phase=1, direction="z")
        if perc.percolates:
            tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity
            X_features.append([vf, surface_area])
            y_target.append(tau)

X_features = np.array(X_features)
y_target = np.array(y_target)

print(f"Successfully generated {len(y_target)} valid samples.")

In [ ]:
print("\n--- Step 2: Training the AI Surrogate Model ---")
X_train, X_test, y_train, y_test = train_test_split(X_features, y_target, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
t0 = time.time()
y_pred = model.predict(X_test)
t_predict = time.time() - t0

r2 = r2_score(y_test, y_pred)
print(f"Model R^2 Score: {r2:.3f}")
print(f"Time to predict {len(X_test)} structures: {t_predict:.4f} seconds")

# Plot True vs Predicted
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, color='#1f77b4', s=50, alpha=0.8)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', lw=2, label="Perfect Prediction")
plt.xlabel("True Tortuosity (OpenImpala)")
plt.ylabel("Predicted Tortuosity (Random Forest)")
plt.title("AI Surrogate Model Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()